In [ ]:
import sys
from tqdm import tqdm

import numpy as np
import seaborn as sns
from scipy.stats import rankdata

from ephyslib import preprocessing
from ephyslib import crossvalidation

from macaquethings.data_util.load_data import load_data
from macaquethings.data_util.process_data import process_data
from macaquethings.rdm_util import get_rdm_design_sort_indices

from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedShuffleSplit,
    cross_validate,
)
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

from macaquethings.plotting.default_styles import *
from macaquethings.plotting.util import legend_without_duplicate_labels

import matplotlib.pyplot as plt
import seaborn as sns

figure_style()  # set consistent plotting defaults for all figs


# Load Data

In [ ]:
monkey = "monkeyF"

if monkey == "monkeyN":
    # --- data config
    data_cfg = dict(
        monkey="monkeyN",
        labels="category",
        baseline=0,
        session_ids=np.array([0, 3, 4, 5]),
        array_indices=np.arange(16) + 1,
        rois=np.array([3]),
        good_channel_threshold=1.5,
        session_ids_for_channel_selection=np.array([0,3,4,5]),
        neural_data="mua",
        dataset='allMUA'
    )
    
    decode_cfg = dict(
        standardize_data=1,
        trial_averaged=False,  # must be false, for compatibility with data preprocessing
        avg_time=10,  # window size for averaging in time
    )
    
    peak_times = np.array([62, 112])

else:
    # --- data config
    data_cfg = dict(
        monkey="monkeyF",
        labels="category",
        baseline=0,
        session_ids=np.array([0, 1, 2, 3, 4, 5]),
        array_indices=np.arange(16) + 1,
        rois=np.array([3]),
        good_channel_threshold=1.5,
        session_ids_for_channel_selection=np.array([0,1,2,3,4,5]),
        neural_data="mua",
        dataset='allMUA'
    )
    
    decode_cfg = dict(
        standardize_data=1,
        trial_averaged=False,  # must be false, for compatibility with data preprocessing
        avg_time=10,  # window size for averaging in time
    )
    
    peak_times = np.array([74, 120])

time, X, y, groups, im_number, sess_idx, info, data_strs, h5_handle = load_data(
    data_cfg, root='..'
)
roi_str, arr_str, sess_str = data_strs
X, groups = process_data(X, sess_idx, im_number, groups, decode_cfg)

# Select Interval and Set up GridSearch

In [ ]:
import tempfile
from joblib import Memory
from pathlib import Path
import shutil


decoding_ts = np.arange(50, 250, 10)
tmask = np.isin(time, decoding_ts)
Xsel = X[..., tmask]


param_grid = {
    "ridgeclassifier__alpha": np.logspace(0, 8, 32),
}

def setup(memory=False, ncomp=None):
    seed = 42
    
    cv = crossvalidation.ImageStratifiedShuffleSplit(5, random_state=seed)
    clf = RidgeClassifier(
        random_state=seed, max_iter=10000, class_weight="balanced"
    )
    model = make_pipeline(
        PCA(ncomp), 
        clf,
        memory='./tmp_spatiotemporal' if memory else None
    )
    
    grid_search = GridSearchCV(
        model,
        param_grid,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise",
    )
    return grid_search
    


# Decode all single time bins

In [ ]:
time_scores = []
for i in tqdm(range(Xsel.shape[2])):
    grid_search = setup()
    Xt = Xsel[..., i]
    grid_search.fit(Xt, y, groups=groups)

    n_splits = grid_search.n_splits_
    best_idx = grid_search.best_index_
    # collect per-fold test scores
    fold_scores = [
        grid_search.cv_results_[f'split{i}_test_score'][best_idx]
        for i in range(n_splits)
    ]

    fold_scores = np.array(fold_scores)
    assert fold_scores.mean() == grid_search.best_score_
    time_scores.append(fold_scores)

time_scores = np.array(time_scores)


In [ ]:
plt.figure(figsize=QUARTER_PANEL_SIZE)
plt.plot(decoding_ts, time_scores.mean(axis=1))
plt.show()

# Decode from Trajectories

In [ ]:
perfs = dict()

## 99% explained variance

In [ ]:
# fit for trajectories

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

Xflat = Xsel.reshape((Xsel.shape[0], -1))

grid_search = setup(memory=True, ncomp=.99)
grid_search.fit(Xflat, y, groups=groups)
print(grid_search.best_score_)
n_splits = grid_search.n_splits_
best_idx = grid_search.best_index_
# collect per-fold test scores
fold_scores = [
    grid_search.cv_results_[f'split{i}_test_score'][best_idx]
    for i in range(n_splits)
]

fold_scores = np.array(fold_scores)
assert fold_scores.mean() == grid_search.best_score_

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

perfs['spatiotemporal 0.99'] = fold_scores

## Match number of channels

In [ ]:
shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

Xflat = Xsel.reshape((Xsel.shape[0], -1))

grid_search = setup(memory=True, ncomp=Xsel.shape[1])
grid_search.fit(Xflat, y, groups=groups)
print(grid_search.best_score_)
n_splits = grid_search.n_splits_
best_idx = grid_search.best_index_
# collect per-fold test scores
fold_scores = [
    grid_search.cv_results_[f'split{i}_test_score'][best_idx]
    for i in range(n_splits)
]

fold_scores = np.array(fold_scores)
assert fold_scores.mean() == grid_search.best_score_

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

perfs['spatiotemporal n_chans'] = fold_scores

## 90% Explained Variance

In [ ]:
shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

Xflat = Xsel.reshape((Xsel.shape[0], -1))

grid_search = setup(memory=True, ncomp=.9)
grid_search.fit(Xflat, y, groups=groups)
print(grid_search.best_score_)
n_splits = grid_search.n_splits_
best_idx = grid_search.best_index_
# collect per-fold test scores
fold_scores = [
    grid_search.cv_results_[f'split{i}_test_score'][best_idx]
    for i in range(n_splits)
]

fold_scores = np.array(fold_scores)
assert fold_scores.mean() == grid_search.best_score_

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

perfs['spatiotemporal 0.9'] = fold_scores

## Concatenate Feedforward Interaction Peak Times

In [ ]:

tmask = np.isin(time, peak_times)
Xsel_twopeak = X[..., tmask]
print(Xsel_twopeak.shape)

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

Xflat_twopeak = Xsel_twopeak.reshape((Xsel_twopeak.shape[0], -1))

grid_search = setup(memory=True, ncomp=None)
grid_search.fit(Xflat_twopeak, y, groups=groups)
print(grid_search.best_score_)
n_splits = grid_search.n_splits_
best_idx = grid_search.best_index_
# collect per-fold test scores
fold_scores = [
    grid_search.cv_results_[f'split{i}_test_score'][best_idx]
    for i in range(n_splits)
]

fold_scores = np.array(fold_scores)
assert fold_scores.mean() == grid_search.best_score_

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

perfs['inter-area peaks'] = fold_scores
 

## Average Feedforard Decoding

In [ ]:
decoding_ts_ff_avg = np.arange(70, 171, 10)
tmask = np.isin(time, decoding_ts_ff_avg)
Xsel_ff_avg = X[..., tmask].mean(axis=-1)
print(Xsel_ff_avg.shape)

shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

grid_search = setup(memory=True, ncomp=None)
grid_search.fit(Xsel_ff_avg, y, groups=groups)
print(grid_search.best_score_)
n_splits = grid_search.n_splits_
best_idx = grid_search.best_index_
# collect per-fold test scores
fold_scores = [
    grid_search.cv_results_[f'split{i}_test_score'][best_idx]
    for i in range(n_splits)
]

fold_scores = np.array(fold_scores)
assert fold_scores.mean() == grid_search.best_score_


shutil.rmtree('./tmp_spatiotemporal', ignore_errors=True)

perfs['feedforward avg.'] = fold_scores
 


In [ ]:
perfs

# Save Results

In [ ]:
import os
from os import path
import pickle

results = {
    'time': decoding_ts,
    'single bin scores': time_scores,
    'spatiotemporal scores': perfs,
    'data_cfg': data_cfg,
    'peak_times': peak_times
}

resdir = 'spatiotemporal_decoding_mua_chan'
run_folder = path.join(resdir, f"results_{data_cfg['monkey']}_roi_{str(data_cfg['rois'])}_peaktimes{str(peak_times)}".replace('[', '').replace(']', '').replace(' ', '_'))
os.makedirs(run_folder, exist_ok=True)

print(run_folder)

with open(path.join(run_folder, 'results.pkl'), 'wb') as f:
    pickle.dump(results, f)
 